<a href="https://colab.research.google.com/github/Tanvilakhani/Deep-learning/blob/main/Section1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import cv2 as cv
import numpy as np
import math

#image displaying libraries
import pylab
from matplotlib import pyplot as plt
from IPython.display import HTML
from IPython.display import clear_output
from base64 import b64encode

#make generated images appear inline below the code
%matplotlib inline

#set the displayed image size
pylab.rcParams['figure.figsize'] = (10.0, 8.0)

# Download example videos
!wget https://raw.githubusercontent.com/Jamesrogers221194/AINT-Files/3f989bb044250abdc3c25ccc5d4c10744e649369/AINT515/Coursework01/Video1%20for%20Vision%20CW.mp4
!wget https://raw.githubusercontent.com/Jamesrogers221194/AINT-Files/3f989bb044250abdc3c25ccc5d4c10744e649369/AINT515/Coursework01/Video2%20for%20Vision%20CW.mp4

--2025-04-28 16:42:56--  https://raw.githubusercontent.com/Jamesrogers221194/AINT-Files/3f989bb044250abdc3c25ccc5d4c10744e649369/AINT515/Coursework01/Video1%20for%20Vision%20CW.mp4
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 786074 (768K) [application/octet-stream]
Saving to: ‘Video1 for Vision CW.mp4’

Video1 for Vision C 100%[===================>] 767.65K  --.-KB/s    in 0.06s   

2025-04-28 16:42:56 (13.2 MB/s) - ‘Video1 for Vision CW.mp4’ saved [786074/786074]

--2025-04-28 16:42:56--  https://raw.githubusercontent.com/Jamesrogers221194/AINT-Files/3f989bb044250abdc3c25ccc5d4c10744e649369/AINT515/Coursework01/Video2%20for%20Vision%20CW.mp4
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.

In [ ]:

import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
from base64 import b64encode
from IPython.display import HTML, clear_output

#Load input video
videoInput = cv.VideoCapture('Video1 for Vision CW.mp4')
#videoInput = cv.VideoCapture('Video2 for Vision CW.mp4') #uncomment to load the second video
if (videoInput.isOpened() == False):
  print("Error opening video stream or file")

#Setup output video writer
videoOutput = cv.VideoWriter('output.avi',cv.VideoWriter_fourcc(*'MJPG'), 30, (int(videoInput.get(cv.CAP_PROP_FRAME_WIDTH)),int(videoInput.get(cv.CAP_PROP_FRAME_HEIGHT))))
# Manually specify the hue values here
background_hue = 100
droplet_hue = 20

# Hue range
hue_range = 7

# Area thresholds for circle detection (adjust as needed)
min_area = 100
max_area = 10000
circularity_threshold = 0.7

# Ray casting parameters
num_rays = 100
max_ray_length = 60
step_size = 1

# Define the ROI for wrap detection
# Format: [x_start, y_start, width, height]
roi = [80, 30, 400, 100]

# Define the vertical line position (x-coordinate)
line_x = 100

# Variables to track contour crossing
crossed_centroids = 0
centroid_positions = {}  # Dictionary to track centroids by their ID
next_centroid_id = 0
processed_centroids = set()

# Get frame dimensions
ret, first_frame = videoInput.read()
if ret:
    frame_height, frame_width = first_frame.shape[:2]
    line_start = (line_x, 0)
    line_end = (line_x, frame_height)
    # Reset the video capture to the beginning
    videoInput.set(cv.CAP_PROP_POS_FRAMES, 0)
else:
    print("Could not read the first frame")
    exit()

#==================================Main Program Loop================================
frame_count = 0
while(1):
    #grab a frame, break the loop if there are no frames left
    ret, frame = videoInput.read()
    if(ret == False):
        break

    frame_count += 1

    # Convert to HSV color space
    hsv_frame = cv.cvtColor(frame, cv.COLOR_BGR2HSV)

    # Convert to grayscale for edge detection
    gray_frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    # Create masks for background and droplet using the specified hue values
    background_mask = cv.inRange(hsv_frame[:,:,0], background_hue-hue_range, background_hue+hue_range)
    droplet_mask = cv.inRange(hsv_frame[:,:,0], droplet_hue-hue_range, droplet_hue+hue_range)

    # Apply morphological operations to clean up the mask
    kernel = np.ones((5,5), np.uint8)
    droplet_mask = cv.morphologyEx(droplet_mask, cv.MORPH_OPEN, kernel)
    droplet_mask = cv.morphologyEx(droplet_mask, cv.MORPH_CLOSE, kernel)

    # Find contours in the droplet mask
    contours, _ = cv.findContours(droplet_mask, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

    # Create a copy of the original frame for drawing
    result_frame = frame.copy()

    # Apply Gaussian blur to the grayscale image to reduce noise
    blurred = cv.GaussianBlur(gray_frame, (5, 5), 0)

    # Draw the ROI rectangle
    x, y, w, h = roi
    cv.rectangle(result_frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

    # Draw the vertical line
    cv.line(result_frame, line_start, line_end, (0, 0, 255), 2)

    # Store current centroids for this frame
    current_centroids = []

    # Filter contours based on area and circularity
    for contour in contours:
        # Calculate area of the contour
        area = cv.contourArea(contour)

        # Filter by area
        if area < min_area or area > max_area:
            continue

        # Calculate perimeter
        perimeter = cv.arcLength(contour, True)

        # Calculate circularity: 4*pi*area/perimeter^2
        # A perfect circle has a circularity of 1.0
        if perimeter > 0:
            circularity = 4 * np.pi * area / (perimeter * perimeter)

            # Filter by circularity
            if circularity > circularity_threshold:
                # Draw the contour in green (inner droplet)
                cv.drawContours(result_frame, [contour], -1, (0, 255, 0), 2)

                # Calculate and draw center point
                M = cv.moments(contour)
                if M["m00"] != 0:
                    cX = int(M["m10"] / M["m00"])
                    cY = int(M["m01"] / M["m00"])

                    # Add to current centroids list
                    current_centroids.append((cX, cY))

                    # Mark the centroid with a color based on its position relative to the line
                    if cX < line_x:
                        cv.circle(result_frame, (cX, cY), 5, (255, 0, 0), -1)  # Blue if left of line
                    else:
                        cv.circle(result_frame, (cX, cY), 5, (0, 0, 255), -1)  # Red if right of line

                    # Draw centroid ID
                    centroid_id = None
                    min_distance = float('inf')
                    matched_pos = None

                    # Try to match this centroid to existing ones
                    for cid, positions in centroid_positions.items():
                        last_pos = positions[-1]
                        dist = np.sqrt((cX - last_pos[0])**2 + (cY - last_pos[1])**2)

                        # If close enough, consider it the same centroid
                        if dist < 30 and dist < min_distance:  # 30 pixel threshold
                            min_distance = dist
                            centroid_id = cid
                            matched_pos = last_pos

                    # If no match, assign a new ID
                    if centroid_id is None:
                        centroid_id = next_centroid_id
                        next_centroid_id += 1
                        centroid_positions[centroid_id] = [(cX, cY)]
                    else:
                        # Check if this centroid has crossed the line
                        if (matched_pos[0] < line_x and cX >= line_x) or (matched_pos[0] >= line_x and cX < line_x):
                            if centroid_id not in processed_centroids:
                                crossed_centroids += 1
                                processed_centroids.add(centroid_id)

                        # Update position
                        centroid_positions[centroid_id].append((cX, cY))


                    # Check if the center point is within the ROI
                    in_roi = (x <= cX <= x+w) and (y <= cY <= y+h)

                    # Only perform wrap detection if the center is in the ROI
                    if in_roi:
                        # Store boundary points found by ray casting
                        boundary_points = []

                        # Cast rays from the center in different directions
                        for angle in range(0, 360, int(360/num_rays)):
                            # Convert angle to radians
                            rad = np.deg2rad(angle)

                            # Calculate ray direction vector
                            dx = np.cos(rad)
                            dy = np.sin(rad)

                            # Variables to track darkest point along ray
                            min_intensity = 255
                            darkest_x, darkest_y = cX, cY

                            # Traverse the ray from inside to outside
                            for r in range(10, max_ray_length, step_size):  # Start a bit away from center
                                # Current point coordinates
                                x_ray = int(cX + dx * r)
                                y_ray = int(cY + dy * r)

                                # Check if point is within image bounds
                                if 0 <= x_ray < frame.shape[1] and 0 <= y_ray < frame.shape[0]:
                                    # Get pixel intensity
                                    intensity = blurred[y_ray, x_ray]

                                    # If this is the darkest point so far, update
                                    if intensity < min_intensity:
                                        min_intensity = intensity
                                        darkest_x, darkest_y = x_ray, y_ray
                                else:
                                    # Ray has gone outside the image
                                    break

                            # Add the darkest point to boundary points
                            if min_intensity < 250:  # Ensure we found a significantly dark point
                                boundary_points.append([darkest_x, darkest_y])
                                # Draw the darkest point
                                # Draw the darkest point (Yellow Circle) with a smaller radius
                                yellow_radius = 2
                                cv.circle(result_frame, (darkest_x, darkest_y), yellow_radius, (0, 255, 255), -1)  # Yellow circle


    # Display the count of crossed centroids
    cv.putText(result_frame, f"Droplet Count: {crossed_centroids}", (160, 25),
           cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 0), 2)

    # Write the frame with circular contours highlighted
    videoOutput.write(result_frame)

# Display output video
videoOutput.release()
print("Loading Video...")
!ffmpeg -i output.avi output.mp4 -hide_banner -loglevel error -y
clear_output()
HTML(f"""<video width=1300 controls autoplay loop><source src="{"data:video/mp4;base64," + b64encode(open('output.mp4', "rb").read()).decode()}" type="video/mp4"></video>""")